# Kaggle Runner: Pointwise PURS (MovieLens-1M)

This notebook runs your updated binary pointwise PURS pipeline end-to-end on Kaggle.

What it does:
- Sets up repository path in a writable location
- Installs dependencies
- Downloads/checks MovieLens-1M into data/raw/ml-1m
- Runs ExperimentRunner with pointwise settings
- Prints HR@10, precision@10, AUC, NDCG@10, unexpectedness@10, coverage@10, serendipity@10

In [ ]:
from pathlib import Path
import os
import subprocess

working_repo = Path('/kaggle/working/recsys')

if not working_repo.exists():
    try:
        print('Cloning recsys into /kaggle/working/recsys ...')
        subprocess.check_call(
            ['git', 'clone', 'https://github.com/codetuscan/recsys.git', str(working_repo)]
        )
    except Exception as exc:
        print('Clone did not complete. Next cell will try existing paths, including /kaggle/input.')
        print('Clone error:', exc)
else:
    print('Using existing clone at', working_repo)

if working_repo.exists():
    os.chdir(working_repo)
    print('Working directory:', Path.cwd())
else:
    print('No working clone yet; continuing to path-detection cell.')

In [ ]:
from pathlib import Path
import os
import sys
import shutil

candidates = [
    Path('/kaggle/working/recsys'),
    Path('/kaggle/input/recsys'),
    Path('/kaggle/input/recsys/recsys'),
    Path.cwd(),
]

repo_root = None
for candidate in candidates:
    if (candidate / 'config' / 'base_config.py').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the recsys repository. Put it in /kaggle/working/recsys or /kaggle/input/recsys.'
    )

if str(repo_root).startswith('/kaggle/input'):
    writable_root = Path('/kaggle/working/recsys')
    if not writable_root.exists():
        shutil.copytree(repo_root, writable_root)
    repo_root = writable_root

os.chdir(repo_root)
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

print('Repository root:', repo_root)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys
from pathlib import Path

requirements_file = Path('requirements-kaggle.txt')
if not requirements_file.exists():
    requirements_file = Path('requirements.txt')

print('Installing dependencies from', requirements_file)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)])
print('Dependency install done.')

In [ ]:
from pathlib import Path
import shutil

raw_dir = Path('data/raw')
ml1m_dir = raw_dir / 'ml-1m'
required_files = ['ratings.dat', 'movies.dat', 'users.dat']
optional_files = ['README']

def has_required_files(folder: Path) -> bool:
    return all((folder / name).exists() for name in required_files)

if has_required_files(ml1m_dir):
    print('MovieLens-1M already available at', ml1m_dir)
else:
    input_root = Path('/kaggle/input')
    candidate_dirs = []

    if input_root.exists():
        for ratings_path in input_root.glob('**/ratings.dat'):
            folder = ratings_path.parent
            if has_required_files(folder):
                candidate_dirs.append(folder)

    if not candidate_dirs:
        raise FileNotFoundError(
            'Could not find MovieLens-1M in Kaggle Input. Add a dataset with ratings.dat, movies.dat, and users.dat.'
        )

    src_dir = sorted(candidate_dirs, key=lambda p: len(str(p)))[0]
    print('Using Kaggle Input dataset from', src_dir)

    raw_dir.mkdir(parents=True, exist_ok=True)
    ml1m_dir.mkdir(parents=True, exist_ok=True)

    for name in required_files + optional_files:
        src = src_dir / name
        dst = ml1m_dir / name
        if src.exists():
            shutil.copy2(src, dst)

for name in required_files:
    path = ml1m_dir / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required file after copy: {path}')

print('MovieLens-1M is ready at', ml1m_dir)

## Run Mode
- Set `SMOKE_RUN = True` for fast validation
- Set `SMOKE_RUN = False` for full baseline (100 epochs)

In [ ]:
from datetime import datetime

SMOKE_RUN = True

subset = 0.02 if SMOKE_RUN else None
epochs = 1 if SMOKE_RUN else 100
run_tag = 'smoke' if SMOKE_RUN else 'full'
exp_name = f'kaggle_purs_pointwise_{run_tag}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print('SMOKE_RUN:', SMOKE_RUN)
print('subset:', subset)
print('epochs:', epochs)
print('experiment:', exp_name)

In [ ]:
from recsys.config import load_config
from recsys.experiments.experiment_runner import ExperimentRunner

config = load_config('kaggle')

# Model/task selection
config.model.model_name = 'purs'
config.data.dataset_name = 'movielens-1m'
config.data.positive_rating_threshold = 3.5

# Baseline hyperparameters
config.model.epochs = epochs
config.model.batch_size = 256
config.model.learning_rate = 0.01
config.model.embedding_dim = 128
config.model.gru_hidden_dim = 128
config.model.dropout = 0.1

# Data/eval settings
config.data.data_subset = subset
config.experiment.k_values = [10]

# Runtime controls
config.experiment.experiment_name = exp_name
config.model.num_workers = 2

runner = ExperimentRunner(config)
results = runner.run()
results

In [ ]:
metric_keys = [
    'best_ndcg@10',
    'hr@10',
    'precision@10',
    'auc',
    'ndcg@10',
    'unexpectedness@10',
    'coverage@10',
    'serendipity@10',
]

print('Final metric summary')
for key in metric_keys:
    if key in results:
        print(f'{key}: {results[key]:.6f}')

print('\nLog directory:', runner.metrics_logger.get_experiment_dir())